# Notebook 1 — Empresas Selecionadas para Análise

**Objetivo:** Extrair da bronze as empresas que atendem aos critérios de elegibilidade para análise financeira e persistir na camada Silver.  
**Input:** `layer_01_bronze.cad_cia_aberta`  
**Output:** `layer_02_silver.n0_empresas_selecionadas`

## 1. Importando Bibliotecas

In [1]:
import pandas as pd
import datetime
import logging
import os
from sqlalchemy import create_engine, text, inspect
from urllib.parse import quote_plus
from dotenv import load_dotenv

In [2]:
# Carrega variáveis de ambiente
load_dotenv()

True

In [3]:
# Definindo a função para criar a engine do banco de dados
def create_db_engine():
        user = quote_plus(os.getenv('DB_USER', 'postgres'))
        password = quote_plus(os.getenv('DB_PASS', 'password'))
        host = os.getenv('DB_HOST', 'localhost')
        port = os.getenv('DB_PORT', '5432')
        dbname = os.getenv('DB_NAME', 'data_lake')
        
        connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        return create_engine(connection_str)

In [4]:
# Efetivamente criando a engine
engine = create_db_engine()

## 2. Query de Seleção — critérios de elegibilidade das empresas

> Os filtros abaixo definem o universo de ~214 empresas que o pipeline vai processar de ponta a ponta.  
> Qualquer alteração aqui se propaga automaticamente para todos os notebooks subsequentes.

In [5]:
query_empresas_selecionadas = '''
    SELECT *
    FROM layer_01_bronze.cad_cia_aberta
    WHERE "SIT" = 'ATIVO'                         -- apenas empresas em situação ativa na CVM
    AND "TP_MERC" = 'BOLSA'                       -- obrigatório: sem cotação, P/L e EV/EBITDA são impossíveis
    AND "SIT_EMISSOR" = 'FASE OPERACIONAL'        -- exclui pré-operacionais e empresas em liquidação
    AND "SETOR_ATIV" NOT LIKE '%Emp. Adm. Part%' -- holdings: estrutura contábil incompatível com o Golden Schema
    AND "SETOR_ATIV" NOT LIKE '%Banc%'            -- bancos: regime prudencial com contabilidade diferente
    AND "SETOR_ATIV" NOT LIKE '%Segurad%'         -- seguradoras: mesma incompatibilidade estrutural
    AND "SETOR_ATIV" NOT LIKE '%Financeira%'      -- financeiras: idem
    AND "SETOR_ATIV" NOT LIKE '%Securitiz%'       -- securitizadoras: idem
    AND "SETOR_ATIV" NOT LIKE '%Adm.%Imóv%'      -- administradoras de imóveis: idem
'''

## 3. Extração e Escrita na Silver

In [6]:
# 1. Extração e Consolidação em Memória
with engine.connect() as conn:
    df_empresas = pd.read_sql(
        text(query_empresas_selecionadas), 
        con=conn
    )

In [7]:
df_empresas.head()

,CNPJ_CIA,DENOM_SOCIAL,DENOM_COMERC,DT_REG,DT_CONST,DT_CANCEL,MOTIVO_CANCEL,SIT,DT_INI_SIT,CD_CVM,...,CEP_RESP,DDD_TEL_RESP,TEL_RESP,DDD_FAX_RESP,FAX_RESP,EMAIL_RESP,CNPJ_AUDITOR,AUDITOR,metadata_data_carga,metadata_arquivo_origem
0,12.528.708/0001-07,AERIS IND. E COM. DE EQUIP. PARA GER. DE ENG. ...,nan,2020-11-09,2020-08-17,nan,nan,ATIVO,2020-11-09,25283,...,4533014.0,85.0,981981880.0,nan,nan,ri@aerisenergy.com.br,54.276.936/0001-79,BDO RCS AUDITORES INDEPENDENTES - SOCIEDADE SI...,2025-12-18 10:18:07.244387,cad_cia_aberta.csv
1,10.338.320/0001-00,AFLUENTE TRANSMISSÃO DE ENERGIA ELETRICA S/A,AFLUENTE TRANSMISSÃO DE ENERGIA ELETRICA S/A,2010-09-24,2008-08-18,nan,nan,ATIVO,2010-09-24,22179,...,22210030.0,21.0,32359828.0,nan,nan,ri@neoenergia.com,49.928.567/0001-11,DELOITTE TOUCHE TOHMATSU AUDITORES INDEPENDENT...,2025-12-18 10:18:07.244387,cad_cia_aberta.csv
2,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,2016-10-26,1992-08-05,nan,nan,ATIVO,2016-10-26,24058,...,4006050.0,11.0,949806361.0,nan,nan,ri@allianca.com,40.262.602/0001-31,"BKR - LOPES, MACHADO AUDITORES",2025-12-18 10:18:07.244387,cad_cia_aberta.csv
3,60.537.263/0001-66,"ALLPARK EMPREENDIMENTOS, PARTICIPAÇÕES E SERVI...","ALLPARK EMPREENDIMENTOS, PARTICIPAÇÕES E SERVI...",2020-05-13,1989-04-19,nan,nan,ATIVO,2020-05-13,24953,...,4543000.0,11.0,21618099.0,nan,nan,ri@estapar.com.br,61.562.112/0001-20,PRICEWATERHOUSECOOPERS AUDITORES INDEPENDENTES...,2025-12-18 10:18:07.244387,cad_cia_aberta.csv
4,61.079.117/0001-05,ALPARGATAS SA,ALPARGATAS,1977-07-20,1907-04-03,nan,nan,ATIVO,1977-07-20,10456,...,4794000.0,11.0,45697322.0,nan,nan,ri@alpargatas.com,61.562.112/0001-20,PRICEWATERHOUSECOOPERS AUDITORES INDEPENDENTES...,2025-12-18 10:18:07.244387,cad_cia_aberta.csv


In [8]:
df_empresas.head(1).to_dict()

{'CNPJ_CIA': {0: '12.528.708/0001-07'},
 'DENOM_SOCIAL': {0: 'AERIS IND. E COM. DE EQUIP. PARA GER. DE ENG. S.A.'},
 'DENOM_COMERC': {0: 'nan'},
 'DT_REG': {0: '2020-11-09'},
 'DT_CONST': {0: '2020-08-17'},
 'DT_CANCEL': {0: 'nan'},
 'MOTIVO_CANCEL': {0: 'nan'},
 'SIT': {0: 'ATIVO'},
 'DT_INI_SIT': {0: '2020-11-09'},
 'CD_CVM': {0: '25283'},
 'SETOR_ATIV': {0: 'Máquinas, Equipamentos, Veículos e Peças'},
 'TP_MERC': {0: 'BOLSA'},
 'CATEG_REG': {0: 'Categoria A'},
 'DT_INI_CATEG': {0: '2020-11-09'},
 'SIT_EMISSOR': {0: 'FASE OPERACIONAL'},
 'DT_INI_SIT_EMISSOR': {0: '2010-08-17'},
 'CONTROLE_ACIONARIO': {0: 'PRIVADO'},
 'TP_ENDER': {0: 'SEDE'},
 'LOGRADOURO': {0: 'Rodovia CE-155'},
 'COMPL': {0: 'km. 2'},
 'BAIRRO': {0: 'CIPP'},
 'MUN': {0: 'CAUCAIA'},
 'UF': {0: 'CE'},
 'PAIS': {0: 'BRASIL'},
 'CEP': {0: '61642000.0'},
 'DDD_TEL': {0: '85.0'},
 'TEL': {0: '34572200.0'},
 'DDD_FAX': {0: '85.0'},
 'FAX': {0: '34572200.0'},
 'EMAIL': {0: 'ri@aerisenergy.com.br'},
 'TP_RESP': {0: 'DIRETOR 

In [9]:
df_empresas.to_sql(
    name='n0_empresas_selecionadas',
    schema='layer_02_silver',
    con=engine,
    if_exists='replace',  # recria a tabela a cada execução — garante idempotência
    index=False,
    chunksize=10000,      # escrita em lotes para não sobrecarregar a conexão
    method='multi'        # insert múltiplo: mais eficiente que um INSERT por linha
)

214